In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import IPython
js_code = """
setInterval(function() {
  var run = document.querySelector('#top-toolbar run-button');
  console.log('keeping alive');
}, 60000);
"""
IPython.display.display(IPython.display.Javascript(js_code))

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import pandas as pd
movie_csv = pd.read_csv('/content/drive/MyDrive/csv/IMDB_Dataset.csv')
print(movie_csv.head())
print(movie_csv.shape)
print(movie_csv.columns)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)
Index(['review', 'sentiment'], dtype='object')


In [ ]:
movie_csv = movie_csv.dropna()
movie_csv['review'] = movie_csv['review'].str.replace(r'<.*?>', '', regex=True)
movie_csv['review'] = movie_csv['review'].str.replace(r'[^a-zA-Z\s]', '', regex=True)
movie_csv['review'] = movie_csv['review'].str.lower()
movie_csv['review'] = movie_csv['review'].str.strip()
movie_csv['sentiment'] = movie_csv['sentiment'].map({'positive': 1, 'negative': 0})

print(movie_csv.head(3))
print(movie_csv.isnull().sum())

                                              review  sentiment
0  one of the other reviewers has mentioned that ...          1
1  a wonderful little production the filming tech...          1
2  i thought this was a wonderful way to spend ti...          1
review       0
sentiment    0
dtype: int64


In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

tokens = tokenizer(
    "this movie was amazing",
    max_length=128,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
print(tokens)

{'input_ids': tensor([[ 101, 2023, 3185, 2001, 6429,  102,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0,

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    movie_csv['review'].tolist(),
    movie_csv['sentiment'].tolist(),
    test_size=0.2,
    random_state=42
)

print(f"Train size: {len(X_train)}")
print(f"Test size:  {len(X_test)}")

Train size: 40000
Test size:  10000


In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, reviews, labels):
        self.reviews = reviews
        self.labels  = labels

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        tokens = tokenizer(
            self.reviews[idx],
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      tokens['input_ids'].squeeze(),
            'attention_mask': tokens['attention_mask'].squeeze(),
            'label':          self.labels[idx]
        }

In [ ]:
train_dataset = IMDBDataset(X_train, y_train)
test_dataset  = IMDBDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 2500
Test batches:  625


In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel

class SentimentClassifier(nn.Module):
    def __init__(self):
        super(SentimentClassifier, self).__init__()

        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):

        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = output.pooler_output
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)

model  = SentimentClassifier()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using: {device}")
print("Model ready!")
print("Model ready!")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using: cuda
Model ready!
Model ready!


In [ ]:
loss_fxn=nn.CrossEntropyLoss()
optimizer=torch.optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = model(input_ids, attention_mask)
        loss = loss_fxn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss  += loss.item()
        predictions  = torch.argmax(outputs, dim=1)
        correct     += (predictions == labels).sum().item()
        total       += labels.size(0)

    train_acc  = correct / total
    train_loss = total_loss / len(train_loader)

    #test
    model.eval()
    test_loss = 0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            loss    = loss_fxn(outputs, labels)

            test_loss    += loss.item()
            predictions   = torch.argmax(outputs, dim=1)
            test_correct += (predictions == labels).sum().item()
            test_total   += labels.size(0)

    test_acc  = test_correct / test_total
    test_loss = test_loss / len(test_loader)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Epoch 1/3 | Train Loss: 0.3078 | Train Acc: 0.8665 | Test Loss: 0.2507 | Test Acc: 0.8925
Epoch 2/3 | Train Loss: 0.1816 | Train Acc: 0.9302 | Test Loss: 0.2573 | Test Acc: 0.8911
Epoch 3/3 | Train Loss: 0.0928 | Train Acc: 0.9672 | Test Loss: 0.3649 | Test Acc: 0.8827


In [14]:
from google.colab import files
import shutil

# 1. Save model
torch.save(model.state_dict(), 'best_model.pth')

# 2. Save tokenizer
tokenizer.save_pretrained('tokenizer/')

# 3. Download both
files.download('best_model.pth')
shutil.make_archive('tokenizer', 'zip', 'tokenizer/')
files.download('tokenizer.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
def predict(text):
    model.eval()

    tokens = tokenizer(
        text,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids      = tokens['input_ids'].to(device)
    attention_mask = tokens['attention_mask'].to(device)

    with torch.no_grad():
        outputs    = model(input_ids, attention_mask)
        prediction = torch.argmax(outputs, dim=1).item()

    return "Positive" if prediction == 1 else "Negative"

print(predict("This movie was very good!"))
print(predict("Worst film I have ever seen"))
print(predict("It was okay nothing special"))

Positive
Negative
Positive
